1)Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")   # suppress deprecation / tie-weight warnings

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

print("Libraries loaded successfully.")

Libraries loaded successfully.


2)Model Loading

In [2]:
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading model: {MODEL_NAME}")
print("(First run downloads ~350 MB — subsequent runs use local cache)\n")

# Load tokenizer — converts text ↔ token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token   # DialoGPT has no pad token; reuse EOS

# Load causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, tie_word_embeddings=False)
model.eval()   # inference mode — disables dropout

# GenerationConfig object avoids deprecated keyword-mixing warning
gen_config = GenerationConfig(
    max_new_tokens=100,
    do_sample=True,
    temperature=0.75,
    top_p=0.92,
    repetition_penalty=1.3,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print("Model loaded and ready.")

Loading model: microsoft/DialoGPT-medium
(First run downloads ~350 MB — subsequent runs use local cache)



Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and ready.


3) User Input Handling

In [3]:
def generate_response(user_message, chat_history_ids=None):
    """
    Generate a chatbot reply for a given user message.

    Parameters
    ----------
    user_message     : str           — the latest user input
    chat_history_ids : torch.Tensor  — encoded history from previous turns

    Returns
    -------
    reply            : str           — the bot's reply text
    chat_history_ids : torch.Tensor  — updated history (pass to next call)
    """
    # Encode user message + EOS separator
    new_input_ids = tokenizer.encode(
        user_message + tokenizer.eos_token,
        return_tensors="pt",
    )

    # Concatenate with history; trim to last 800 tokens to avoid context overflow
    if chat_history_ids is not None:
        history_trimmed = chat_history_ids[:, -800:]
        bot_input_ids = torch.cat([history_trimmed, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate reply (no_grad saves memory during inference)
    with torch.no_grad():
        output_ids = model.generate(bot_input_ids, generation_config=gen_config)

    # Decode only the newly generated tokens
    reply_ids  = output_ids[:, bot_input_ids.shape[-1]:]
    reply_text = tokenizer.decode(reply_ids[0], skip_special_tokens=True).strip()

    if not reply_text:
        reply_text = "Could you tell me more? I want to make sure I understand."

    return reply_text, output_ids

4)Response Generation

In [4]:
print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

chat_history_ids = None   # starts empty; grows each turn

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Thank you for chatting! Goodbye!")
        break

    response, chat_history_ids = generate_response(user_input, chat_history_ids)
    print(f"Chatbot: {response}\n")

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hello! How are you?!

You: what is artificial intelligence?
Chatbot: A person with some kind of technology in their head that can sense, or have a memory of things. The same as our brain. What are your thoughts on it?

You: who created python?
Chatbot: No one really knows who made it but they certainly did not use it for anything. It's just an example of the idea behind it to be honest...

You: quit
Chatbot: Thank you for chatting! Goodbye!


In [4]:
import nbformat

# Read your current notebook
nb = nbformat.read("/content/your_notebook.ipynb", as_version=4)

# Remove widget metadata (this is the problem)
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# Clean each cell
for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

# Save fixed notebook
nbformat.write(nb, "/content/final_notebook.ipynb")

print("✅ Fixed notebook created")